## Fase 6B: Extração de Informações Relevantes para o Modelo de IA (Parte 1)

Neste notebook, vamos realizar a primeira parte da extração de informações relevantes dos dados pré-processados na Fase 6A para o desenvolvimento do modelo de IA para previsão de produtividade agrícola.

In [ ]:
# Configuração do ambiente
import sys
sys.path.append('../../')
import setup_notebook
setup_notebook.setup_environment()

%matplotlib inline

## Introdução

Na Fase 6A, realizamos o pré-processamento dos dados de NDVI/EVI e produtividade agrícola. Agora, vamos extrair informações relevantes desses dados para o desenvolvimento do modelo de IA para previsão de produtividade agrícola.

Neste notebook (Parte 1), vamos focar nas seguintes tarefas:

1. Carregamento dos dados pré-processados
2. Definição das variáveis-chave para o modelo
3. Análise inicial da relação entre NDVI/EVI e produtividade agrícola

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from scipy import stats

# Configurar o estilo dos gráficos
plt.style.use('fivethirtyeight')
sns.set(style="whitegrid")

## 1. Carregamento dos Dados Pré-processados

Vamos carregar os dados pré-processados na Fase 6A.

In [ ]:
# Verificar se os arquivos existem
if os.path.exists('../../sprint2/assets/ndvi_mensal_nova_friburgo_preprocessado.csv') and \
   os.path.exists('../../sprint2/assets/ndvi_mensal_teresopolis_preprocessado.csv') and \
   os.path.exists('../../sprint2/assets/evi_produtividade_2017.csv'):
    # Carregar os dados pré-processados
    df_ndvi_nf = pd.read_csv('../../sprint2/assets/ndvi_mensal_nova_friburgo_preprocessado.csv')
    df_ndvi_t = pd.read_csv('../../sprint2/assets/ndvi_mensal_teresopolis_preprocessado.csv')
    df_evi_prod = pd.read_csv('../../sprint2/assets/evi_produtividade_2017.csv')
    
    print("Dados pré-processados carregados com sucesso!")
else:
    # Se os arquivos não existirem, carregar os dados originais
    print("Os arquivos de dados pré-processados não foram encontrados. Carregando os dados originais...")
    
    # Carregar os dados de NDVI/EVI
    df_ndvi_nf = pd.read_csv('../../assets/ndvi_mensal_nova_friburgo.csv')
    df_ndvi_t = pd.read_csv('../../assets/ndvi_mensal_teresopolis.csv')
    
    # Carregar os dados de produtividade agrícola
    df_prod_combinados = pd.read_csv('../../assets/dados_produtividade_combinados.csv')
    
    # Filtrar os dados de produtividade para o ano de 2017
    df_prod_2017 = df_prod_combinados[df_prod_combinados['Ano'] == 2017]
    
    # Agrupar os dados de produtividade por município e tipo de lavoura
    df_prod_grouped = df_prod_2017.groupby(['Município', 'Tipo de Lavoura'])['Produtividade (t/ha)'].mean().reset_index()
    
    # Verificar os anos disponíveis nos dados de NDVI/EVI
    print("Anos disponíveis nos dados de Nova Friburgo:")
    print(df_ndvi_nf['Ano'].unique())
    print("\nAnos disponíveis nos dados de Teresópolis:")
    print(df_ndvi_t['Ano'].unique())
    
    # Filtrar os dados de NDVI/EVI para o ano de 2017
    df_ndvi_nf_2017 = df_ndvi_nf[df_ndvi_nf['Ano'] == 2017]
    df_ndvi_t_2017 = df_ndvi_t[df_ndvi_t['Ano'] == 2017]
    
    # Verificar se existem dados para 2017
    if len(df_ndvi_nf_2017) > 0 and len(df_ndvi_t_2017) > 0:
        # Calcular a média anual do EVI para cada município em 2017
        evi_nf_2017 = df_ndvi_nf_2017['EVI_mean'].mean()
        evi_t_2017 = df_ndvi_t_2017['EVI_mean'].mean()
        
        print(f"\nMédia anual do EVI para Nova Friburgo em 2017: {evi_nf_2017:.4f}")
        print(f"Média anual do EVI para Teresópolis em 2017: {evi_t_2017:.4f}")
    else:
        # Se não existirem dados para 2017, usar o último ano disponível
        ultimo_ano_nf = df_ndvi_nf['Ano'].max()
        ultimo_ano_t = df_ndvi_t['Ano'].max()
        
        print(f"\nNão existem dados para 2017. Usando o último ano disponível:")
        print(f"Nova Friburgo: {ultimo_ano_nf}")
        print(f"Teresópolis: {ultimo_ano_t}")
        
        # Filtrar os dados para o último ano disponível
        df_ndvi_nf_2017 = df_ndvi_nf[df_ndvi_nf['Ano'] == ultimo_ano_nf]
        df_ndvi_t_2017 = df_ndvi_t[df_ndvi_t['Ano'] == ultimo_ano_t]
        
        # Calcular a média anual do EVI para cada município no último ano disponível
        evi_nf_2017 = df_ndvi_nf_2017['EVI_mean'].mean()
        evi_t_2017 = df_ndvi_t_2017['EVI_mean'].mean()
        
        print(f"\nMédia anual do EVI para Nova Friburgo no último ano disponível: {evi_nf_2017:.4f}")
        print(f"Média anual do EVI para Teresópolis no último ano disponível: {evi_t_2017:.4f}")
    
    # Criar um dataframe com os dados de EVI e produtividade
    data = {
        'Município': ['Nova Friburgo', 'Nova Friburgo', 'Teresópolis', 'Teresópolis'],
        'Tipo de Lavoura': ['Temporária', 'Permanente', 'Temporária', 'Permanente'],
        'EVI_mean': [evi_nf_2017, evi_nf_2017, evi_t_2017, evi_t_2017]
    }
    df_evi_prod = pd.DataFrame(data)
    
    # Mesclar com os dados de produtividade
    df_evi_prod = pd.merge(df_evi_prod, df_prod_grouped, on=['Município', 'Tipo de Lavoura'])
    
    print("Dados carregados e processados manualmente.")

# Exibir informações sobre os dataframes
print("\nInformações sobre os dados de NDVI/EVI de Nova Friburgo:")
print(f"Número de registros: {df_ndvi_nf.shape[0]}")
print(f"Número de colunas: {df_ndvi_nf.shape[1]}")
print(f"Colunas: {', '.join(df_ndvi_nf.columns)}")

print("\nInformações sobre os dados de EVI e produtividade:")
print(f"Número de registros: {df_evi_prod.shape[0]}")
print(f"Número de colunas: {df_evi_prod.shape[1]}")
print(f"Colunas: {', '.join(df_evi_prod.columns)}")

## 2. Definição das Variáveis-chave para o Modelo

Vamos definir as variáveis-chave que serão utilizadas no modelo de IA para previsão de produtividade agrícola.

In [ ]:
# Converter a coluna de data para o formato datetime (se existir)
if 'Data' in df_ndvi_nf.columns:
    df_ndvi_nf['Data'] = pd.to_datetime(df_ndvi_nf['Data'])
    df_ndvi_t['Data'] = pd.to_datetime(df_ndvi_t['Data'])
else:
    # Criar a coluna de data a partir do ano e mês
    df_ndvi_nf['Data'] = pd.to_datetime(df_ndvi_nf['Ano'].astype(str) + '-' + df_ndvi_nf['Mês'].astype(str) + '-01')
    df_ndvi_t['Data'] = pd.to_datetime(df_ndvi_t['Ano'].astype(str) + '-' + df_ndvi_t['Mês'].astype(str) + '-01')

# Função para classificar o mês em estação do ano
def get_season(month):
    if month in [12, 1, 2]:
        return 'Verão'
    elif month in [3, 4, 5]:
        return 'Outono'
    elif month in [6, 7, 8]:
        return 'Inverno'
    else:  # month in [9, 10, 11]
        return 'Primavera'

# Adicionar coluna de estação do ano
df_ndvi_nf['Estação'] = df_ndvi_nf['Mês'].apply(get_season)
df_ndvi_t['Estação'] = df_ndvi_t['Mês'].apply(get_season)

# Exibir as primeiras linhas dos dataframes com a coluna de estação
print("Primeiras linhas do dataframe de Nova Friburgo com a coluna de estação:")
print(df_ndvi_nf[['Ano', 'Mês', 'Estação', 'EVI_mean', 'SG_mean']].head())

print("\nPrimeiras linhas do dataframe de Teresópolis com a coluna de estação:")
print(df_ndvi_t[['Ano', 'Mês', 'Estação', 'EVI_mean', 'SG_mean']].head())

In [ ]:
# Agrupar por ano e estação para calcular estatísticas sazonais
df_ndvi_nf_seasonal = df_ndvi_nf.groupby(['Ano', 'Estação']).agg({
    'EVI_mean': ['mean', 'min', 'max', 'std'],
    'SG_mean': ['mean', 'min', 'max', 'std']
}).reset_index()

# Renomear as colunas para facilitar o acesso
df_ndvi_nf_seasonal.columns = ['Ano', 'Estação', 'EVI_mean_mean', 'EVI_mean_min', 'EVI_mean_max', 'EVI_mean_std',
                             'SG_mean_mean', 'SG_mean_min', 'SG_mean_max', 'SG_mean_std']

# Fazer o mesmo para Teresópolis
df_ndvi_t_seasonal = df_ndvi_t.groupby(['Ano', 'Estação']).agg({
    'EVI_mean': ['mean', 'min', 'max', 'std'],
    'SG_mean': ['mean', 'min', 'max', 'std']
}).reset_index()

# Renomear as colunas para facilitar o acesso
df_ndvi_t_seasonal.columns = ['Ano', 'Estação', 'EVI_mean_mean', 'EVI_mean_min', 'EVI_mean_max', 'EVI_mean_std',
                           'SG_mean_mean', 'SG_mean_min', 'SG_mean_max', 'SG_mean_std']

# Verificar os anos disponíveis nos dados sazonais
print("Anos disponíveis nos dados sazonais de Nova Friburgo:")
print(df_ndvi_nf_seasonal['Ano'].unique())
print("\nAnos disponíveis nos dados sazonais de Teresópolis:")
print(df_ndvi_t_seasonal['Ano'].unique())

# Verificar se existem dados para 2017
ano_referencia = 2017
if (ano_referencia in df_ndvi_nf_seasonal['Ano'].values) and (ano_referencia in df_ndvi_t_seasonal['Ano'].values):
    # Exibir os dados sazonais para o ano de 2017
    print(f"\nDados sazonais de Nova Friburgo para {ano_referencia}:")
    print(df_ndvi_nf_seasonal[df_ndvi_nf_seasonal['Ano'] == ano_referencia])
    
    print(f"\nDados sazonais de Teresópolis para {ano_referencia}:")
    print(df_ndvi_t_seasonal[df_ndvi_t_seasonal['Ano'] == ano_referencia])
    
    # Se existirem dados para 2017, usar 2017 como ano de referência
    ano_referencia_nf = ano_referencia
    ano_referencia_t = ano_referencia
else:
    # Se não existirem dados para 2017, usar o último ano disponível
    ultimo_ano_nf = df_ndvi_nf_seasonal['Ano'].max()
    ultimo_ano_t = df_ndvi_t_seasonal['Ano'].max()
    
    print(f"\nNão existem dados sazonais para {ano_referencia}. Usando o último ano disponível:")
    print(f"Nova Friburgo: {ultimo_ano_nf}")
    print(f"Teresópolis: {ultimo_ano_t}")
    
    # Exibir os dados sazonais para o último ano disponível
    print(f"\nDados sazonais de Nova Friburgo para {ultimo_ano_nf}:")
    print(df_ndvi_nf_seasonal[df_ndvi_nf_seasonal['Ano'] == ultimo_ano_nf])
    
    print(f"\nDados sazonais de Teresópolis para {ultimo_ano_t}:")
    print(df_ndvi_t_seasonal[df_ndvi_t_seasonal['Ano'] == ultimo_ano_t])
    
    # Atualizar o ano de referência
    ano_referencia_nf = ultimo_ano_nf
    ano_referencia_t = ultimo_ano_t

In [ ]:
# Vamos criar um dataframe com as features sazonais para o ano de referência
# e mesclar com os dados de produtividade

# Filtrar os dados sazonais para o ano de referência
df_ndvi_nf_seasonal_ref = df_ndvi_nf_seasonal[df_ndvi_nf_seasonal['Ano'] == ano_referencia_nf].copy()
df_ndvi_t_seasonal_ref = df_ndvi_t_seasonal[df_ndvi_t_seasonal['Ano'] == ano_referencia_t].copy()

# Adicionar coluna de município
df_ndvi_nf_seasonal_ref['Município'] = 'Nova Friburgo'
df_ndvi_t_seasonal_ref['Município'] = 'Teresópolis'

# Concatenar os dataframes
df_seasonal_ref = pd.concat([df_ndvi_nf_seasonal_ref, df_ndvi_t_seasonal_ref])

# Pivotar o dataframe para ter uma linha por município e colunas para cada estação
df_seasonal_pivot = df_seasonal_ref.pivot_table(
    index='Município',
    columns='Estação',
    values=['EVI_mean_mean', 'SG_mean_mean']
).reset_index()

# Renomear as colunas
df_seasonal_pivot.columns = ['Município'] + [
    f"{col[0]}_{col[1]}" for col in df_seasonal_pivot.columns.values[1:]
]

# Exibir o dataframe pivotado
print(f"Dataframe pivotado com features sazonais para o ano de referência:")
print(df_seasonal_pivot)

## 3. Análise Inicial da Relação entre NDVI/EVI e Produtividade Agrícola

Vamos analisar a relação entre os índices NDVI/EVI e a produtividade agrícola.

In [ ]:
# Mesclar com os dados de produtividade
# Primeiro, vamos calcular a média de produtividade por município
df_prod_mean = df_evi_prod.groupby('Município')['Produtividade (t/ha)'].mean().reset_index()

# Mesclar com as features sazonais
df_features = pd.merge(df_seasonal_pivot, df_prod_mean, on='Município')

# Exibir o dataframe com as features e a produtividade
print("Dataframe com features sazonais e produtividade:")
print(df_features)

In [ ]:
# Vamos visualizar a relação entre as features sazonais e a produtividade
# Criar um gráfico de barras para as features sazonais
plt.figure(figsize=(14, 8))

# Selecionar as colunas de EVI
evi_cols = [col for col in df_features.columns if col.startswith('EVI_mean_')]

# Criar um dataframe para o gráfico
df_plot = df_features[['Município'] + evi_cols].melt(
    id_vars='Município',
    var_name='Estação',
    value_name='EVI'
)

# Limpar os nomes das estações
df_plot['Estação'] = df_plot['Estação'].str.replace('EVI_mean_', '')

# Verificar os valores no dataframe para o gráfico
print("Valores no dataframe para o gráfico:")
print(df_plot)

# Plotar o gráfico de barras
sns.barplot(x='Estação', y='EVI', hue='Município', data=df_plot)
plt.title(f'EVI Médio por Estação e Município ({ano_referencia})')
plt.xlabel('Estação')
plt.ylabel('EVI Médio')
plt.legend(title='Município')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Vamos visualizar a relação entre a produtividade e o EVI médio anual
plt.figure(figsize=(10, 6))

# Calcular o EVI médio anual
df_features['EVI_mean_anual'] = df_features[evi_cols].mean(axis=1)

# Plotar o gráfico de dispersão
sns.scatterplot(x='EVI_mean_anual', y='Produtividade (t/ha)', data=df_features, s=100)

# Adicionar rótulos aos pontos
for i, row in df_features.iterrows():
    plt.text(row['EVI_mean_anual'], row['Produtividade (t/ha)'], row['Município'], fontsize=12)

plt.title(f'Relação entre EVI Médio Anual e Produtividade ({ano_referencia})')
plt.xlabel('EVI Médio Anual')
plt.ylabel('Produtividade (t/ha)')
plt.grid(True)
plt.tight_layout()
plt.show()

## Conclusão da Parte 1

Neste notebook, realizamos a primeira parte da extração de informações relevantes dos dados pré-processados. Definimos as variáveis-chave para o modelo e realizamos uma análise inicial da relação entre os índices NDVI/EVI e a produtividade agrícola.

Principais observações:

1. Criamos features sazonais a partir dos dados mensais de NDVI/EVI, agrupando os meses em estações do ano.
2. Analisamos a relação entre as features sazonais e a produtividade agrícola.
3. Visualizamos a relação entre o EVI médio anual e a produtividade.

Na próxima parte (Fase 6B - Parte 2), continuaremos a análise com foco na identificação dos períodos críticos de crescimento da cultura e na extração de features temporais mais específicas.